# 31 - Automated Cleaning Tool
This notebook reflects the logic produced in `21-automated_cleaning_tool.py`, explaining what each cell's syntax/logic aims to produce or develop. In this particular notebook, we will be cleaning our example `11-classic_titanic_dataset.csv` to prepare it for the predictive modeler, `22-interactive_predictive_modeler.py` down the road.

In [11]:
from pathlib import Path
import pandas as pd
import unicodedata
import re

### Defining Date Parsing & Numeric Coercing

In [2]:
# Prior to our input section, we'll simply define our preluding functions, `parse_dates` and `coerce_numeric`:
def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            try:
                df[col] = pd.to_datetime(df[col], errors='raise') # Try to parse as date
            except Exception: # If it raises an exception,
                pass # Forgeeet about it!
    return df
# This logic aims to ensure no matter HOW many date columns you have in whichever dataset YOU'd like to use, they will all be parsed correctly.
def coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors='ignore')
    return df
# And this does the same but with numeric columns, tries to force them to be numeric. 


### Input Section

In [4]:
# Now, we will ask them how they'd like to set up their cleaning configuration.
# For each prompt, the end user will effectively build their own list:
a = []
a_flag = True
while a_flag == True:
    a_input = input("Enter the name of a date column you would like to parse: ")
    a.append(a_input)
    a_continue = input("Continue? Y/N: ")
    if a_continue == "N":
        a_flag = False
# For a, the end user selects date columns they'd like to parse.
# Because of how we defined our function, they CAN select a non-date column. 
# It will just ignore the exception raised and move to the next option in their list.

b = []
b_flag = True
while b_flag == True:
    b_input = input("Enter the name of a numeric column you would like to configure: ")
    b.append(b_input)
    b_continue = input("Continue? Y/N: ")
    if b_continue == "N":
        b_flag = False
# For b, the user will enter any numeric columns they'd like to configure.

c = []
c_flag = True
while c_flag == True:
    c_input = input("Enter the name of a column you would prefer to drop: ")
    c.append(c_input)
    c_continue = input("Continue? Y/N: ")
    if c_continue == "N":
        c_flag = False
# For c, the user will enter columns they'd like to drop.

# d (below) is unique because it has to be a dictionary, not a list.
# That's because we're asking the end user if they'd like to CHANGE a column name.
# So they can say, "I want to change this to that, this to that, this to that," etc.
d = {}
d_flag = True
while d_flag == True:
    key_input = input("If there is a column you'd like to rename, provide said column's current name: ")
    value_input = input("Provide the name you'd prefer the column to have: ")
    d[key_input] = value_input
    d_continue = input("Continue? Y/N: ")
    if d_continue == "N":
        d_flag = False
# Here, the user will enter the name's they'd like to change.

f = []
f_flag = True
while f_flag == True:
    f_input = input("If there is a column you'd like to normalize entirely, provide such a column's name here: ")
    f.append(f_input)
    f_continue = input("Continue? Y/N: ")
    if f_continue == "N":
        f_flag = False
# For f, the user will enter any columns they'd like to "normalize" entirely.

Enter the name of a date column you would like to parse:  
Continue? Y/N:  N
Enter the name of a numeric column you would like to configure:  Fare
Continue? Y/N:  N
Enter the name of a column you would prefer to drop:  Ticket
Continue? Y/N:  Y
Enter the name of a column you would prefer to drop:  Cabin
Continue? Y/N:  N
If there is a column you'd like to rename, provide said column's current name:  SibSp
Provide the name you'd prefer the column to have:  Family
Continue? Y/N:  N
If there is a column you'd like to normalize entirely, provide such a column's name here:  
Continue? Y/N:  N


### Configuration

In [5]:
# Now we will set up the configuration:
cleaning_config = {
    "date_columns": a,
    "numeric_columns": b,
    "drop_columns": c,
    "rename_columns": d,
    "normalize_columns": f
}

In [6]:
# And define how we will apply this configuration to the cleaning script it's given:
def apply_config(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if cleaning_config["drop_columns"]: # If the user added columns to drop,
        df = df.drop(columns=cleaning_config["drop_columns"], errors="ignore") # Cut them out
    if cleaning_config["rename_columns"]: # If the user added columns to rename,
        df = df.rename(columns=cleaning_config["rename_columns"]) # Rename them via the dict object
    for col in cleaning_config["date_columns"]: # If they gave date columns,
        if col in df.columns: # If column IS still a column,
            df[col] = pd.to_datetime(df[col], errors='coerce') # Parse to date
    for col in cleaning_config["numeric_columns"]: # If they added numeric columns,
        if col in df.columns: # If column IS still a column,
            df[col] = pd.to_numeric(df[col], errors='coerce') # Convert it to numeric
    return df

In [7]:
# Lastly, we will define what "normalization" means:
def normalize_string(s: str) -> str:
    if not isinstance(s, str): # If its NOT a string,
        return s # Don't worry about it.
    s = unicodedata.normalize("NFKD", s) # Breaks down accented or special characters
    s = s.lower() # Make it all lowercase
    s = re.sub(r"[-_/]", "", s) # Just SCRAPS hyphens, underscores, and slashe
    s = re.sub(r"\s+", "", s) # Scraps spaces, tabs, and 'new lines'
    s = s.strip() # And, of course, removes leading/trailing whitespace
    return s

def normalize_text_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(normalize_string)
    return df
# Applies our new function when a column is selected and appended to the normalizer

### Defining the `basic_clean`

In [8]:
# This is just going to take whatever dataframe it's been given, make adjustments,
# Then APPLY the configuration steps. Afterwards, it will drop duplicates, drop NAs,
# and handle dates/numeric columns.
def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns =(
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    df = apply_config(df)
    if cleaning_config["normalize_columns"]:
        df = normalize_text_columns(df, cleaning_config["normalize_columns"])
    df = df.drop_duplicates()
    df = df.dropna(axis=1, how='all')
    df = parse_dates(df)
    df = coerce_numeric(df)
    return df


### Defining the `inspect`

In [9]:
# This is a simple cell that defines inspect as a function that reads the dataframe,
# prints its name, provides its shape and column types, and returns its missing values.
def inspect(df: pd.DataFrame, name: str = "DataFrame") -> None:
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isna().sum())

### Output Section

In [12]:
# Finally, we have our output section.
raw_filename = raw_input
clean_filename = clean_input
# This top section refers back to the raw input path we were given at the beginning,
# And points the CLEAN output back in the direction of the clean path we were given.
print("Loading raw data...")
df_raw = pd.read_csv(raw_filename)
inspect(df_raw, name = "Raw data")
print("Applying basic cleaning steps...")
df_clean = basic_clean(df_raw)
print("Saving cleaned data...")
(df_clean).to_csv(clean_input, index=False)
print("Done.")
# And gives some fancy output as it goes along.

Loading raw data...

--- Raw data ---
Shape: (891, 12)

Columns:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
Applying basic cleaning steps...
Saving cleaned data...
Done.
